In [ ]:
!pip install --quiet docling

: 

In [ ]:
import os 
import requests

In [ ]:
import os

local_file = r"C:\Ask My Paper\Data\Copy of AI Engineering-25-125.pdf"

if os.path.exists(local_file):
    print(f"PDF found: {local_file}")
else:
    print("PDF file not found.")

Now that the file is loaded in our local folder, we can use Docling to parse
and analyze it.

In [ ]:
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat


pipeline_options = PdfPipelineOptions()
pipeline_options.generate_picture_images = True
res = DocumentConverter( format_options={ InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options), }).convert(local_file)
doc = res.document

In [ ]:
table = doc.tables[1]
table_df = table.export_to_dataframe()
table_df

In [ ]:
print(table_df.shape)

In [ ]:
def format_context_item(item, max_rows=100): 
    """
    Formats context items dynamically, handling strings, DataFrames, 
    and nested JSON with safety checks for production.
    """ 
    # 1. Handle standard text chunks 
    if isinstance(item, str): 
        return f"- FACT: {item}" 
     
    # 2. Handle DataFrames (dynamic schema) 
    if isinstance(item, pd.DataFrame): 
        # If the table is too long, we truncate it but keep the headers 
        if len(item) > max_rows: 
            item = item.head(max_rows) 
            truncation_note=f"\n[Note: Table truncated to first {max_rows} rows]" 
        else: 
            truncation_note = "" 
             
        return f"""- DATAFRAME (Columns: {list(item.columns)}):
                {item.to_json(orient='records', indent=2)}{truncation_note}""" 
 
    # 3. Handle generic structured data (JSON/Dict) 
    try: 
        serialized = json.dumps(item, indent=2) 
        return f"- STRUCTURED DATA:\n{serialized}" 
    except (TypeError, ValueError): 
        # Fallback if the data isn't JSON-serializable 
        return f"- DATA (Raw): {str(item)}" 
 
 
formatted_context = "\n".join(format_context_item(c) for c in context)
prompt = f"""
You are an assistant for question-answering tasks. Use the following pieces of 
retrieved context to answer the question. If you don't know the answer, just 
say 
that you don't know. 
<question>
{question}
</question> 
<context>
{formatted_context}
</context> 
Answer:
"""